In [1]:
from cs336_basics.tokenizer import  Tokenizer
tokenizer = Tokenizer.from_files('../tests/fixtures/gpt2_vocab.json','../tests/fixtures/gpt2_merges.txt',['<|endoftext|>','<|endoftext|><|endoftext|>'])
# print(tokenizer.vocab_rev)
# print(tokenizer.merges)
import itertools
import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
# Create a console handler
console_handler = logging.StreamHandler()
# Create a formatting layout
formatter = logging.Formatter('%(name)s - %(levelname)s - %(asctime)s - %(message)s')
console_handler.setFormatter(formatter)

# Add the handler to your logger
logger.addHandler(console_handler)

cs336_basics.tokenizer - INFO - 2026-10-06 10:09:51,706 - Creating a Tokenizer
cs336_basics.tokenizer - INFO - 2026-10-06 10:09:51,831 - Tokenizer created.
cs336_basics.tokenizer - DEBUG - 2026-10-06 10:09:51,838 - Vocab successfully loaded
cs336_basics.tokenizer - DEBUG - 2026-10-06 10:09:51,848 - Merges successfully loaded
cs336_basics.tokenizer - DEBUG - 2026-10-06 10:09:51,850 - Special tokens are ['<|endoftext|>', '<|endoftext|><|endoftext|>']


In [10]:
from collections.abc import Iterator
from collections.abc import Iterable
from cs336_basics.bpe import get_pretokens
get_pretokens("a\n b",special_tokens=None)
get_pretokens("a\n",special_tokens=None)
get_pretokens(" b",special_tokens=None)
# def count_up_to(max_value:int)->Iterator[int]:
#     count = 1
#     while count <= max_value:
#         yield count
#         count += 1
#
# # Initialize the generator object
# counter = count_up_to(3)

#
# # Accessing values manually
# print(next(counter))  # Output: 1
# print(next(counter))  # Output: 2
# print(next(counter))  # Output: 3

# Calling next() again will raise a StopIteration exception,
# signaling that the generator is empty.

cs336_basics.bpe - DEBUG - 2026-10-06 10:26:22,523 - Pretokens are ['a', '\n', ' b']
cs336_basics.bpe - DEBUG - 2026-10-06 10:26:22,526 - Pretokens are ['a', '\n']
cs336_basics.bpe - DEBUG - 2026-10-06 10:26:22,529 - Pretokens are [' b']


[' b']

In [7]:
import regex as re
from collections.abc import Iterator
from collections.abc import Iterable

special_tokens=['<|endoftext|>']
special_tokens = sorted(special_tokens, key=lambda k: len(k), reverse=True)
pretoken_pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
regex_pattern = "(" + "|".join(map(re.escape, special_tokens)) + ")"

def encode_iterable(iterable:Iterable[str])->Iterator[int]:

    for full_string in iterable:
        partitions = re.split(regex_pattern, full_string)
        logger.debug("Partitions are %s",partitions)
        pretokens = []
        for partition in partitions:
            logger.debug("Partition is %s",partition)
            if partition in special_tokens:
                pretokens.append(partition)
            else:
                pretokens.extend(re.findall(pretoken_pattern, partition))
        logger.debug("Pretokens are %s",pretokens)
        pretokens_encoded:list[int]=[]
        for pretoken in pretokens:
            if len(pretoken)>0:
                if special_tokens is not None and pretoken in special_tokens:
                    st_id = tokenizer.vocab_rev[pretoken.encode("utf-8")]
                    logger.debug("Appending special token ID %d",st_id)
                    pretokens_encoded.append(st_id)
                else:
                    id_list = tokenizer.encode_pretoken(pretoken)
                    logger.debug("Extending token id list %s", id_list)
                    pretokens_encoded.extend(id_list)
        logger.debug("Encoded tokens are %s",pretokens_encoded)
        for encoded in pretokens_encoded:
            yield encoded


filename='../tests/fixtures/tinystories_sample.txt'
fhandle = open(filename,'r')
i=0
# for line in fhandle:
#     print(line)
#     if i==100:
#         break
#     i+=1

counter = encode_iterable(iterable = fhandle)
for i in range(20):
    logger.debug(next(counter))  # Output: 1

fhandle.close()


__main__ - DEBUG - 2026-10-06 10:20:01,615 - Partitions are ['\n']
__main__ - DEBUG - 2026-10-06 10:20:01,617 - Partition is 

__main__ - DEBUG - 2026-10-06 10:20:01,618 - Pretokens are ['\n']
cs336_basics.tokenizer - DEBUG - 2026-10-06 10:20:01,618 - Pretoken representation is [b'\n']
cs336_basics.tokenizer - DEBUG - 2026-10-06 10:20:01,619 - Only one element left
cs336_basics.tokenizer - DEBUG - 2026-10-06 10:20:01,619 - b_array is [b'\n'] Id list is [198]
__main__ - DEBUG - 2026-10-06 10:20:01,619 - Extending token id list [198]
__main__ - DEBUG - 2026-10-06 10:20:01,620 - Encoded tokens are [198]
__main__ - DEBUG - 2026-10-06 10:20:01,623 - 198
__main__ - DEBUG - 2026-10-06 10:20:01,625 - Partitions are ['Once upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he wa

In [2]:


from cs336_basics.io_utils import gpt2_bytes_to_unicode
#teststring = None
teststring = 'Julian'
#teststring = "🙃"
#teststring = '🙃 Julian<|endoftext|> is a boy'

#print(teststring)
encoded = tokenizer.encode(teststring)
logger.debug("teststring is %s and encoded is %s",teststring,encoded)
decoded = tokenizer.decode(encoded)
logger.debug("teststring is %s and decoded is %s",teststring,decoded)
#print(b'<|endoftext|>')

cs336_basics.bpe - DEBUG - 2026-10-05 16:52:49,157 - Special tokens are: ['<|endoftext|>']
cs336_basics.bpe - DEBUG - 2026-10-05 16:52:49,159 - Partitions are ['Julian']
cs336_basics.bpe - DEBUG - 2026-10-05 16:52:49,162 - Pretokens are ['Julian']
cs336_basics.tokenizer - DEBUG - 2026-10-05 16:52:49,163 - Pretoken representation is [b'J', b'u', b'l', b'i', b'a', b'n']
cs336_basics.tokenizer - DEBUG - 2026-10-05 16:52:49,165 - Best candidate is (b'a', b'n') found at index 4
cs336_basics.tokenizer - DEBUG - 2026-10-05 16:52:49,166 - Pretoken representation is [b'J', b'u', b'l', b'i', b'an']
cs336_basics.tokenizer - DEBUG - 2026-10-05 16:52:49,167 - Best candidate is (b'u', b'l') found at index 1
cs336_basics.tokenizer - DEBUG - 2026-10-05 16:52:49,168 - Pretoken representation is [b'J', b'ul', b'i', b'an']
cs336_basics.tokenizer - DEBUG - 2026-10-05 16:52:49,168 - Best candidate is (b'i', b'an') found at index 2
cs336_basics.tokenizer - DEBUG - 2026-10-05 16:52:49,168 - Pretoken represen

In [2]:
import regex as re

special_tokens=["<|endoftext|>", "<|endoftext|><|endoftext|>"]
test_string = "Hello, how <|endoftext|><|endoftext|> are you?<|endoftext|>"

special_tokens =sorted(special_tokens,key=lambda k:len(k),reverse=True)
logger.debug("Special tokens are: %s",special_tokens)
regex_pattern ="("+ "|".join(map(re.escape,special_tokens))+")"

logger.debug(regex_pattern)
partitions = re.split(regex_pattern,test_string)
print(f"Partition {partitions}")
encoded2 = tokenizer.encode(test_string)
encoded2


__main__ - DEBUG - 2026-10-05 17:08:45,139 - Special tokens are: ['<|endoftext|><|endoftext|>', '<|endoftext|>']
__main__ - DEBUG - 2026-10-05 17:08:45,140 - (<\|endoftext\|><\|endoftext\|>|<\|endoftext\|>)
cs336_basics.bpe - DEBUG - 2026-10-05 17:08:45,142 - Special tokens are: ['<|endoftext|><|endoftext|>', '<|endoftext|>']
cs336_basics.bpe - DEBUG - 2026-10-05 17:08:45,143 - Partitions are ['Hello, how ', '<|endoftext|><|endoftext|>', ' are you?', '<|endoftext|>', '']
cs336_basics.bpe - DEBUG - 2026-10-05 17:08:45,144 - Pretokens are ['Hello', ',', ' how', ' ', '<|endoftext|><|endoftext|>', ' are', ' you', '?', '<|endoftext|>']
cs336_basics.tokenizer - DEBUG - 2026-10-05 17:08:45,145 - Pretoken representation is [b'H', b'e', b'l', b'l', b'o']
cs336_basics.tokenizer - DEBUG - 2026-10-05 17:08:45,146 - Best candidate is (b'l', b'l') found at index 2
cs336_basics.tokenizer - DEBUG - 2026-10-05 17:08:45,146 - Pretoken representation is [b'H', b'e', b'll', b'o']
cs336_basics.tokenizer - 

Partition ['Hello, how ', '<|endoftext|><|endoftext|>', ' are you?', '<|endoftext|>', '']


[15496, 11, 703, 220, 50257, 389, 345, 30, 50256]